In [ ]:
from notebookutils import mssparkutils
from datetime import datetime

try:
    batch_id = mssparkutils.widgets.get("batch_id")
except Exception:
    batch_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")

print(f"batch_id = {batch_id}")

In [13]:
from pyspark.sql import functions as F

# 1. Silver transform
BRONZE = "tip_bronze"
SILVER = "tip_silver"

df_tip = spark.table(BRONZE)

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 15, Finished, Available, Finished, False)

In [14]:
base_cols = [
    "user_id",
    "business_id",
    "text",
    "date",
    "compliment_count",
    "_ingest_ts",
    "_source_file",
    "_batch_id"
]

df_tip_silver = df_tip.select(*base_cols)

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 16, Finished, Available, Finished, False)

In [15]:
df_tip_silver = (
    df_tip_silver
    .withColumn("text", F.trim(F.col("text")))
    .withColumn("compliment_count", F.coalesce(F.col("compliment_count"), F.lit(0)).cast("int"))
    
    .filter(F.col("user_id").isNotNull())
    .filter(F.col("business_id").isNotNull())
    .filter(F.col("date").isNotNull())
    .filter(F.col("compliment_count") >= 0)

    .dropDuplicates(["user_id", "business_id", "text", "date"])
)

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 17, Finished, Available, Finished, False)

In [16]:
(
    df_tip_silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(SILVER)
)

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 18, Finished, Available, Finished, False)

In [17]:
df_tip_silver.printSchema()
print("tip_silver rows: ", spark.table("tip_silver").count())

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 19, Finished, Available, Finished, False)

root
 |-- user_id: string (nullable = true)
 |-- business_id: string (nullable = true)
 |-- text: string (nullable = true)
 |-- date: string (nullable = true)
 |-- compliment_count: integer (nullable = false)
 |-- _ingest_ts: timestamp (nullable = true)
 |-- _source_file: string (nullable = true)
 |-- _batch_id: string (nullable = true)

tip_silver rows:  908848


In [18]:
mssparkutils.notebook.exit("SUCCESS")

StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 20, Finished, Available, Finished, False)

ExitValue: SUCCESS

In [19]:
#2. Data Quality Check

df_tip_silver_DQ = spark.table("tip_silver")

print("tip_silver rows = ", df_tip_silver_DQ.count())

dup = (
    df_tip_silver_DQ.groupBy("user_id", "business_id", "text", "date")
    .count()
    .filter(F.col("count") > 1)
    .filter(F.length(F.col("text")) > 0)
    .count()
)
print("duplicate tip rows = ", dup)

df_tip_silver_DQ.select(
    F.sum(F.col("user_id").isNull().cast("int")).alias("null_user_id"),
    F.sum(F.col("business_id").isNull().cast("int")).alias("null_business_id"),
    F.sum(F.col("date").isNull().cast("int")).alias("null_date"),
    F.sum(F.col("compliment_count").isNull().cast("int")).alias("null_compliment_count")
).show()

negative_compliment = df_tip_silver_DQ.filter(F.col("compliment_count") < 0).count()
print("negative compliment_count rows = ", negative_compliment)


StatementMeta(, 79e02224-edf0-4917-99bb-3972fdc20936, 21, Finished, Available, Finished, False)

tip_silver rows =  908848
duplicate tip rows =  0
+------------+----------------+---------+---------------------+
|null_user_id|null_business_id|null_date|null_compliment_count|
+------------+----------------+---------+---------------------+
|           0|               0|        0|                    0|
+------------+----------------+---------+---------------------+

negative compliment_count rows =  0
